In [1]:
import pandas as pd
import os
from collections import defaultdict
from datetime import datetime, timedelta
import re
import numpy as np
from pathlib import Path

os.chdir('/home/nobre/Notebooks/RQAR_2025_book/')

In [18]:
funcoes = {
    
    'MT': rectify_MT,
}

lista_estados = ['MT']

tabela_ids = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/Monitoramento_QAr_BR.csv')
tabela_pols = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/dicionarios/CODIGO_POLUENTES.csv')


In [72]:

estado = 'DF'
  
path = os.getcwd()+'/data/DADOS_BRUTOS/' + estado + '/'

#df_ids = funcoes[estado](path)
    
#create_df_estacao(estado,df_ids)

In [52]:
dict_pols_stat = defaultdict(list)

files = os.listdir(path)

print(files)

for item in files:
    
    estacao = " ".join(item.split('-')[1].split('.')[0].split('_')[0:2])

    print(estacao)

    df = pd.read_excel(path+item)

    df = df.drop(columns=['Nome da estação'])

    lista_pols = set(df['Poluente'])

    for pol in lista_pols:

        df_pol = df[df["Poluente"] == pol]
        
        if pol in ['no2','so2','o3']:

            df_pol = ppb_to_ug(df_pol,pol)

        df_pol_hora = df_pol.groupby(["Ano", "Mes", "Dia", "Hora", "Unidade"])
        
        df_pol_hora = df_pol_hora.filter(lambda g: len(g) >= 9)
        
        df_pol = (
            df_pol_hora.groupby(["Ano", "Mes", "Dia", "Hora", "Unidade"], as_index=False)
                  .agg({"Valor": "mean"})
        )

        df_pol['QAQC_INTERNO'] = None

        df_pol = df_pol.rename(columns={'Ano':'ANO',
                                        'Mes':'MES',
                                        'Dia':'DIA',
                                        'Hora':'HORA',
                                        'Unidade':'UNIDADE',
                                        'Valor':'VALOR'})

        for col in ["ANO", "MES", "DIA", "HORA"]:
            df_pol[col] = pd.to_numeric(df_pol[col], errors="coerce").astype("Int64")
        
        dict_pols_stat[estacao+'_'+pol].append(df_pol)



['dados_monitoramento-Sema.xlsx', 'dados_monitoramento-BEA_CBA_24-25.xlsx', 'dados_monitoramento-BEA_CBA_22-23.xlsx', 'dados_monitoramento-CBM_VG_22-23.xlsx', 'dados_monitoramento-CBM_VG_24-25.xlsx', 'dados_monitoramento-Mae_Bonifacia_22-23.xlsx', 'dados_monitoramento-Mae_Bonifacia_24-25.xlsx', 'dados_monitoramento-UFMT.xlsx']
Sema
BEA CBA
BEA CBA
CBM VG
CBM VG
Mae Bonifacia
Mae Bonifacia
UFMT


In [46]:
def ppb_to_ug(df,pol):

    if pol == 'so2':

        df.loc[df["Unidade"] != "ug/m3", "Valor"] *= 2661260.49/10**6        

    elif pol == 'no2':

        df.loc[df["Unidade"] != "ug/m3", "Valor"] *= 1911038.92/10**6        

    elif pol == 'o3':

        df.loc[df["Unidade"] != "ug/m3", "Valor"] *= 1993889.17/10**6        
    
    df.loc[:, "Unidade"] = "ug/m3"

    return df

In [60]:
dict_pols_MT = {
    'co':'CO',
    'no2': 'NO2',
    'so2': 'SO2',
    'o3': 'O3',
    'pm2p5':'MP25',
    'pm10': 'MP10'
}

dict_formatado = {}

for chave in dict_pols_stat.keys():
    
    lista_dfs = dict_pols_stat[chave]
    
    df = pd.concat(lista_dfs, ignore_index=True)

    df["DATETIME"] = pd.to_datetime(
        df.apply(lambda r: f"{r.ANO}-{r.MES}-{r.DIA} {r.HORA}:00:00", axis=1)
    )
    df = df.set_index("DATETIME")

    df = df.sort_index()
    
    lista_horas = pd.date_range(
        start=df.index.min(), 
        end=df.index.max(), 
        freq='H').strftime('%Y-%m-%d %H:%M:%S').tolist()
    
    if len(lista_horas) != len(df):
        df = df.reindex(pd.DatetimeIndex(lista_horas))

    df['DATETIME'] = df.index

    df = df[['DATETIME','ANO','MES','DIA','HORA','VALOR','UNIDADE','QAQC_INTERNO']]
    
    dict_formatado[chave] = df

primeiros_valores = {}

for chave, df in dict_formatado.items():
    
    if ~df['VALOR'].isna().all() and (df['VALOR'] > 0).any(): 
        
        linha_valida = df[df["VALOR"].notna() & (df["VALOR"] > 0)].iloc[0]
        primeiros_valores[chave] = linha_valida["DATETIME"]

codigo_estacao_MT = {}

for chave in primeiros_valores.keys():
    
    station = chave.split('_')[0]
    data = primeiros_valores[chave]
    
    if station in codigo_estacao_MT:
        if data <= codigo_estacao_MT[station]:
            codigo_estacao_MT[station] = data
    else:
        codigo_estacao_MT[station] = data

sorted_items = sorted(
    codigo_estacao_MT.items(),
    key=lambda x: (x[1], x[0])
)

codigo_estacao_MT = {}
for i, (nome, ts) in enumerate(sorted_items, start=1):
    codigo = f"MT{i:04d}"
    codigo_estacao_MT[nome] = codigo
    
for chave, df in dict_formatado.items():
    
    estacao = codigo_estacao_MT[chave.split('_')[0]]
    
    cod_pol = tabela_pols.loc[tabela_pols['POLUENTE'] == dict_pols_MT[chave.split('_')[-1]], 'COD_POLUENTE'].values[0]
    
    nome_pasta = tabela_pols.loc[tabela_pols['COD_POLUENTE'] == int(cod_pol), 'NOME_PASTA'].values[0]
    
    df.to_csv('/home/nobre/Notebooks/RQAR_2025_book/data/MQAr/'+nome_pasta+'/'+estacao+'RA'+str(cod_pol).zfill(3)+'.csv',index=False)

df_ids = pd.DataFrame({
    'ID_OEMA': codigo_estacao_MT.keys(),
    'ID_MMA':list(codigo_estacao_MT.values())})

return df_ids

/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/2479180702.py:25: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = p

SyntaxError: 'return' outside function (2479180702.py, line 85)

In [61]:
df_ids

,ID_OEMA,ID_MMA
0,Mae Bonifacia,MT0001
1,BEA CBA,MT0002
2,CBM VG,MT0003
3,Sema,MT0004
4,UFMT,MT0005


In [185]:
from datetime import timedelta

def fix_24h(row):
    if isinstance(row, str) and row.startswith("24:"):
        # substitui 24: por 00:
        new_str = row.replace("24:", "00:", 1)
        # converte para datetime
        dt = pd.to_datetime(new_str, errors="coerce")
        # adiciona 1 dia
        if pd.notna(dt):
            dt += timedelta(days=1)
        return dt
    else:
        return pd.to_datetime(row, errors="coerce")



In [70]:
dict_stations_MT = {
        'Sema':'CPA - SEMA - CBA',
        'BEA CBA': 'Dom Aquino - BEA - CBA',
        'CBM VG': 'Água Limpa - CBM - VG',
        'Mae Bonifacia': 'Duque de Caxias - Pq Mãe Bonifácia - CBA',
        'UFMT':'Boa Esperança - UFMT - CBA'
    }
    
df_ids = pd.DataFrame({
    'ID_OEMA': codigo_estacao_MT.keys(),
    'ID_MMA':list(codigo_estacao_MT.values())})


df_ids["ID_OEMA"] = df_ids["ID_OEMA"].replace(dict_stations_MT)


df_ids['ID_OEMA']

0    Duque de Caxias - Pq Mãe Bonifácia - CBA
1                      Dom Aquino - BEA - CBA
2                       Água Limpa - CBM - VG
3                            CPA - SEMA - CBA
4                  Boa Esperança - UFMT - CBA
Name: ID_OEMA, dtype: object

In [71]:
df_ids

,ID_OEMA,ID_MMA
0,Duque de Caxias - Pq Mãe Bonifácia - CBA,MT0001
1,Dom Aquino - BEA - CBA,MT0002
2,Água Limpa - CBM - VG,MT0003
3,CPA - SEMA - CBA,MT0004
4,Boa Esperança - UFMT - CBA,MT0005


In [120]:
estado = 'DF'

In [249]:
path = os.getcwd()+'/data/DADOS_BRUTOS/' + estado + '/'

path = path + 'Monitor Report 2024_FINAL.xlsx'

df = pd.read_excel(path)

df.iloc[1] = df.iloc[1].ffill()

poluentes = ['CO_ppm','NO2_ug/m3','NO_ug/m3','NOx_ug/m3','O3_ug/m3','PM10','PM25','PTS','SO2_ug/m3']

dict_pols = {'CO_ppm':'CO',
             'NO2_ug/m3':'NO2',
             'NO_ug/m3':'NO',
             'NOx_ug/m3':'NOX',
             'O3_ug/m3':'O3',
             'PM10':'MP10',
             'PM25':'MP25',
             'PTS':'PTS',
             'SO2_ug/m3':'SO2'}

df.columns = df.iloc[1]

df = df.drop(index=[0, 1]).reset_index(drop=True)

df = df.rename(columns={'Date Time':'DATETIME'})

estacoes = set(df.columns[1:])

dict_pols_stat = defaultdict(list)

for estacao in estacoes:

    df_estacao = df[["DATETIME",estacao]]

    df_estacao.columns = [df_estacao.columns.tolist()[0]] + df_estacao.iloc[0, 1:].tolist()

    df_estacao = df_estacao.drop(index=[0]).reset_index(drop=True)

    for pol in poluentes:

        if pol in df_estacao.columns:
            
            df_pol = df_estacao[["DATETIME",pol]]

            df_pol['UNIDADE'] = df_pol[pol][0]

            df_pol = df_pol.drop(index=[0]).reset_index(drop=True)

            df_pol = df_pol[df_pol["DATETIME"].astype(str).str.contains(r"\d", na=False)].reset_index(drop=True)

            df_pol['DATETIME'] = df_pol['DATETIME'].apply(fix_24h)

            df_pol = df_pol.rename(columns={pol:'VALOR'})

            df_pol.index = df_pol['DATETIME']

            lista_horas = pd.date_range(
                start=df_pol.index.min(), 
                end=df_pol.index.max(), 
                freq='H').strftime('%Y-%m-%d %H:%M:%S').tolist()
            
            if len(lista_horas) != len(df_pol):
                df_pol = df_pol.reindex(pd.DatetimeIndex(lista_horas))
            
            df_pol['QAQC_INTERNO'] = None
            
            df_pol.insert(1, 'ANO', df_pol.index.year)
            df_pol.insert(2, 'MES', df_pol.index.month)
            df_pol.insert(3, 'DIA', df_pol.index.day)
            df_pol.insert(4, 'HORA', df_pol.index.hour)

            pol = dict_pols[pol]

            df_pol['VALOR'] = pd.to_numeric(df_pol['VALOR'], errors='coerce')

            df_pol = df_pol[['DATETIME','ANO','MES','DIA','HORA','VALOR','UNIDADE','QAQC_INTERNO']] 

            dict_pols_stat[estacao+'_'+pol] = df_pol
            

/tmp/ipykernel_564959/901552261.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pol['UNIDADE'] = df_pol[pol][0]
/tmp/ipykernel_564959/1508039512.py:14: UserWarning: Parsing dates in %d:%M %m/%H/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(row, errors="coerce")
/tmp/ipykernel_564959/901552261.py:57: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  lista_horas = pd.date_range(
/tmp/ipykernel_564959/901552261.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docu

In [258]:
primeiros_valores = {}

for chave, df in dict_pols_stat.items():  
    
    if ~df['VALOR'].isna().all() and (df['VALOR'] > 0).any(): 
        
        linha_valida = df[df["VALOR"].notna() & (df["VALOR"] > 0)].iloc[0]
        primeiros_valores[chave] = linha_valida["DATETIME"]

codigo_estacao_DF = {}

for chave in primeiros_valores.keys():
    
    station = chave.split('_')[0]
    data = primeiros_valores[chave]
    
    if station in codigo_estacao_DF:
        if data <= codigo_estacao_DF[station]:
            codigo_estacao_DF[station] = data
    else:
        codigo_estacao_DF[station] = data

sorted_items = sorted(
    codigo_estacao_DF.items(),
    key=lambda x: (x[1], x[0])
)

codigo_estacao_DF = {}
for i, (nome, ts) in enumerate(sorted_items, start=1):
    codigo = f"DF{i:04d}"
    codigo_estacao_DF[nome] = codigo
    
for chave, df in dict_pols_stat.items():
    
    estacao = codigo_estacao_DF[chave.split('_')[0]]
    
    cod_pol = tabela_pols.loc[tabela_pols['POLUENTE'] == chave.split('_')[-1], 'COD_POLUENTE'].values[0]
    
    nome_pasta = tabela_pols.loc[tabela_pols['COD_POLUENTE'] == int(cod_pol), 'NOME_PASTA'].values[0]
    
    df.to_csv('/home/nobre/Notebooks/RQAR_2025_book/data/MQAr/'+nome_pasta+'/'+estacao+'RA'+str(cod_pol).zfill(3)+'.csv',index=False)

dict_oemas = {
    'Estação CRAS FERCAL': 'Fercal CRAS',
    'Estação Escola':	   'Fercal Escola'}

df_ids = pd.DataFrame({
    'ID_OEMA': codigo_estacao_DF.keys(),
    'ID_MMA':list(codigo_estacao_DF.values())})

df_ids['ID_OEMA'] = df_ids['ID_OEMA'].replace(dict_oema)

return df_ids

In [275]:
df_ufs = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/dicionarios/IBGE_UFS_CODIGOS.csv')
    
cod_uf =  df_ufs.loc[df_ufs['UF'] == uf, 'CODIGOS'].values[0]

print(cod_uf)

if os.path.exists('/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/'+uf+'_estacoes.csv'):

    df_estacao = pd.read_csv('/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/'+uf+'_estacoes.csv')

    df_estacao['ID_MMA'] = df_estacao['ID_OEMA'].map(df_ids.set_index('ID_OEMA')['ID_MMA'])

else:

    colunas = ['ID_OEMA', 'UF', 'ID_MMA', 'COD_UF_IBGE', 'CIDADE', 'CD_MUN',
               'PROPRIETARIO', 'PROP_ENTIDADE', 'OPERADOR', 'OP_ENTIDADE', 'LATITUDE',
               'LONGITUDE', 'MOBILIDADE', 'REALOCACAO', 'MARCA', 'CATEGORIA',
               'FUNCIONAMENTO', 'METODO', 'FINALIDADE', 'POLUENTE',
               'INICIO', 'STATUS', 'FIM', 'CALIBRACAO', 'OBS_CALIBRACAO', 'MONITORAR',
               'FONTE', 'OBS_GERAIS','DADOS_MONITORAMENTO','RECONHECIDA','REP_ESPACIAL_DECLARADA']
    
    df_estacao = pd.DataFrame(columns=colunas)

df_ids = pol_to_station(df_ids)

mapa = dict(zip(df_ids['ID_MMA'], df_ids['POLUENTE']))

df_estacao['POLUENTE'] = df_estacao['ID_MMA'].map(mapa).fillna(df_estacao['POLUENTE'])

#df_estacao = df_estacao.reindex(df_ids.index)

#df_estacao[["ID_MMA", "ID_OEMA", "POLUENTE"]] = df_ids[["ID_MMA", "ID_OEMA", "POLUENTE"]].values

df_estacao.loc[:, "COD_UF_IBGE"] = cod_uf
df_estacao.loc[:, "UF"] = uf
    
#df_estacao.to_csv('/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/'+uf+'_estacoes_teste.csv', index=False)

53


In [276]:
df_estacao['POLUENTE']

0                                 PM2,5
1                                  PM10
2                             MP10,MP25
3                                  PM10
4                                  PM10
5                           MP2,5, MP10
6                           MP2,5, MP10
7                           MP2,5, MP10
8    MP10,NO,CO,PTS,O3,SO2,NO2,NOX,MP25
Name: POLUENTE, dtype: object

In [262]:
def pol_to_station(df_ids):

    base_path = Path('/home/nobre/Notebooks/RQAR_2025_book/data/MQAr/')

    id_to_poluentes = {}
    
    for poluente_dir in base_path.iterdir():
        if poluente_dir.is_dir():
            poluente = poluente_dir.name
    
            arquivos = [arq.stem for arq in poluente_dir.glob("*")]
    
            for id_mma in df_ids["ID_MMA"]:
                if any(str(arq).startswith(id_mma) for arq in arquivos):
                    id_to_poluentes.setdefault(id_mma, []).append(poluente)
    
    df_ids["POLUENTE"] = df_ids["ID_MMA"].map(id_to_poluentes).fillna("").apply(lambda x: ",".join(x) if isinstance(x, list) else "")
    
    return(df_ids)